# VitalFit · Encabezado del Proyecto Integrador 2

**Grupo 3 · Membresias y clases de gimnasio**
Analitica Empresarial Integrada · TECSUP 2026-II · Docente: Pilar Rocio Sayan Mejia

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rociosayan/analitica-empresarial-integrada/blob/main/proyecto02/ENCABEZADO-PMD2-G3-VITALFIT.ipynb)

## Para que sirve este cuaderno

Es **lo primero que ejecutan en la semana 9**, antes de modelar nada. Hace cuatro cosas:

1. lee las ocho tablas del caso desde el repositorio de datos,
2. **estructura la base de datos en DuckDB**, con las ocho tablas como tablas reales,
3. corre diez controles de recepcion para que sepan que los datos llegaron completos,
4. deja preparada la base en un archivo, que es la que usara el asistente de la semana 14.

De la semana 9 a la 15 trabajan sobre esta misma base. No hay que volver a leer los CSV.

> **Un aviso que ahorra una tarde.** En los laboratorios 1 a 4 usaron DuckDB para consultar un
> DataFrame directamente, con `duckdb.sql(df)` y sin crear ninguna base. Aqui se hace el paso
> siguiente: `connect()`, `create table` y guardar la base en un archivo, igual que en el
> **LAB-D14**. Ese paso ya esta escrito mas abajo, no tienen que inventarlo.

## La decision que este proyecto debe sostener

A que socios ofrecer orientacion y como asignar las clases de noviembre de 2026 para mejorar la contribucion esperada y la cobertura de demanda dentro de los recursos disponibles.

Todo lo que hagan de la semana 9 a la 15 existe para poder responder esa pregunta con numeros:
**a que socios ofrecer orientacion y cuantas sesiones programar por franja**, y demostrar que la propuesta es mejor que la politica actual de la empresa.

## Como esta organizado

| Semana | Que se hace | Capacidad de la rubrica |
|---|---|---|
| 9 | Clientes y valor economico | 1. Predice riesgo y demanda con evaluacion honesta (5) |
| 10 | Pronostico de demanda | 1. Predice riesgo y demanda con evaluacion honesta (5) |
| 11 | Optimizacion con PuLP | 2. Optimiza recursos y controla operaciones (5) |
| 12 | Procesos internos | 2. Optimiza recursos y controla operaciones (5) |
| 13 | Gobierno y explicabilidad | 3. Audita el modelo y gobierna los datos (3) |
| 14 | Asistente y checkpoint | 4. Implementa y evalua un asistente analitico (3) |
| 15 | Producto y comunicacion | 5. Integra Colab reproducible y producto desplegado (4) |

Cada seccion de semana trae su consigna escrita dentro. Lean la consigna antes de escribir codigo.

---
## Paso 0 · Parametros del equipo

Lo unico que deben cambiar es `USUARIO`, y solo despues de hacer su *fork* del repositorio de
datos. El fork tiene que quedar **publico**, porque el cuaderno lee los CSV por URL.

In [ ]:
GRUPO   = 'GRUPO_3'
USUARIO = 'Rociosayan'          # <-- cambie esto por SU usuario de GitHub despues del fork
REPO    = 'pmd2-aei-2026ii'
RAMA    = 'main'
CASO    = 'caso3_vitalfit'
SEMILLA = 20261101

# Fecha de decision: la cartera de este corte sirve para DECIDIR a quien contactar.
# No sirve para elegir modelos ni umbrales, porque su etiqueta todavia no existe.
FECHA_DECISION = '2026-11-01'

# Particion temporal del ejercicio principal (cortes del primer domingo de cada mes)
FIN_ENTRENAMIENTO = '2026-04-30'
VALIDACION = ('2026-05-01', '2026-05-31')
PRUEBA     = ('2026-06-01', '2026-06-30')

print(GRUPO, '|', CASO)

In [ ]:
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'duckdb', 'pulp<4', 'statsmodels', 'scikit-learn', 'shap'], check=False)

In [ ]:
import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(SEMILLA)
pd.set_option('display.width', 160, 'display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25})

# Aqui guardan las figuras y las tablas que luego entran al informe y a la app.
SALIDAS = Path('salidas'); SALIDAS.mkdir(exist_ok=True)

print('duckdb', duckdb.__version__, '| pandas', pd.__version__)

---
## Paso 1 · Leer las ocho tablas

Los datos **no estan en este repositorio**. Estan en
`Rociosayan/pmd2-aei-2026ii`, carpeta `casos/caso3_vitalfit/data`, y se leen por URL.

Fijense en una linea de la funcion `leer`: convierte a fecha toda columna que sea una fecha.
Si `fecha_corte` se queda como texto, cualquier filtro por mes o cualquier resta de fechas
falla **sin decir por que**, y es el error que mas tiempo hace perder al principio.

In [ ]:
BASE_URL = f'https://raw.githubusercontent.com/{USUARIO}/{REPO}/{RAMA}/casos/{CASO}/data'

TABLAS = ['clientes', 'actividad', 'cohortes', 'demanda',
          'recursos', 'operaciones', 'politica_actual', 'documentos']

def leer(nombre):
    """Lee una tabla del caso y convierte a fecha las columnas que lo son."""
    df = pd.read_csv(f'{BASE_URL}/{nombre}.csv')
    for col in df.columns:
        if col.startswith('fecha') or col in ('semana', 'seguimiento_fin'):
            df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

datos = {t: leer(t) for t in TABLAS}
resumen = pd.DataFrame({t: {'filas': len(d), 'columnas': d.shape[1]} for t, d in datos.items()}).T
print('Datos leidos desde:', BASE_URL)
resumen

---
## Paso 2 · Estructurar la base de datos

Esto es lo nuevo respecto de los laboratorios 1 a 4. Alla escribian `duckdb.sql(df)` y DuckDB
miraba el DataFrame directamente, sin base. Aqui se abre una **conexion** y las ocho tablas
quedan como tablas de verdad, con nombre propio, consultables con SQL entre ellas.

Por que conviene:

- se pueden cruzar tablas con `join` sin cargar nada en memoria dos veces,
- las vistas que creen en la semana 9 siguen disponibles en la semana 13,
- y el asistente de la semana 14 necesita exactamente esto: una base con tablas y nombres.

`set threads = 1` no es un adorno. Con un solo hilo, las sumas en coma flotante se hacen siempre
en el mismo orden y el cuaderno da **el mismo numero en cada ejecucion**. La rubrica pide un
Colab reproducible.

In [ ]:
con = duckdb.connect()            # base en memoria, vive mientras viva el cuaderno
con.execute('set threads = 1')    # mismo orden de operaciones => mismo resultado cada vez

for t, df in datos.items():
    con.register(f'_{t}', df)                                        # el DataFrame se ve desde SQL
    con.execute(f'create or replace table {t} as select * from _{t}')  # y queda como tabla real

con.execute('show tables').df()

Una primera consulta para comprobar que la base responde. Pide las **reservas**
solicitadas en las ultimas cuatro semanas del historico, agrupadas por linea.

In [ ]:
con.execute('''
    select linea,
           sum(solicitado) as solicitado_4sem,
           sum(atendido)   as atendido_4sem,
           sum(faltante)   as faltante_4sem
    from demanda
    where semana > (select max(semana) - interval 28 day from demanda)
    group by linea
    order by solicitado_4sem desc
''').df()

---
## Paso 3 · Controles de recepcion

Diez comprobaciones. Si alguna sale en `False`, **avisen antes de modelar**: puede ser un dato
del caso que hay que documentar, no necesariamente un error suyo.

Las dos ultimas son las que mas importan para la rubrica, porque demuestran que la particion
temporal esta bien hecha: que la etiqueta del corte de decision todavia no existe, y que hay
casos de las dos clases en entrenamiento, validacion y prueba.

In [ ]:
clientes, actividad, cohortes, demanda = (datos[t] for t in
                                          ['clientes', 'actividad', 'cohortes', 'demanda'])

act_alta = actividad.merge(clientes[['id_cliente', 'fecha_alta']], on='id_cliente')
tramos = {'entrenamiento': ('2000-01-01', FIN_ENTRENAMIENTO),
          'validacion': VALIDACION, 'prueba': PRUEBA}

controles = {
    '1. id_cliente unico en clientes': clientes.id_cliente.is_unique,
    '2. id_evento unico en actividad': actividad.id_evento.is_unique,
    '3. actividad apunta a clientes que existen': actividad.id_cliente.isin(clientes.id_cliente).all(),
    '4. cohortes apunta a clientes que existen': cohortes.id_cliente.isin(clientes.id_cliente).all(),
    '5. clave (id_cliente, fecha_corte) unica': not cohortes.duplicated(['id_cliente', 'fecha_corte']).any(),
    '6. clave (semana, sede, linea) unica': not demanda.duplicated(['semana', 'sede', 'linea']).any(),
    '7. solicitado = atendido + faltante': (demanda.solicitado == demanda.atendido + demanda.faltante).all(),
    '8. ningun evento es anterior al alta del cliente': (act_alta.fecha_evento >= act_alta.fecha_alta).all(),
    '9. la etiqueta del corte de decision aun no existe':
        cohortes.loc[cohortes.fecha_corte == FECHA_DECISION, 'churn'].isna().all(),
    '10. hay churn 0 y 1 en los tres tramos': all(
        cohortes[(cohortes.elegible == 1) & cohortes.fecha_corte.between(a, b)].churn.nunique() == 2
        for a, b in tramos.values()),
}

tabla = pd.Series(controles, name='cumple').to_frame()
print('Controles que NO cumplen:', int((~tabla.cumple).sum()))
tabla

---
## Paso 4 · Como se lee `recursos.csv`

Esta tabla es la que mas confunde, porque **no esta en formato de hoja de calculo**. Viene en
formato largo: una fila por recurso y por linea, y muchas celdas vacias, porque no todos los
recursos tienen margen o tope. Hay dos clases de fila:

- `linea = 'todas'` -> es un **tope del plan** completo (lo que no se puede pasar),
- `linea = '<nombre>'` -> es un **coeficiente de esa linea** (lo que consume o deja cada unidad).

Las dos funciones de abajo le ahorran leer la tabla a mano: `tope()` trae un limite del plan y
`coef()` trae un diccionario `linea -> valor`, que es justo la forma en que PuLP lo necesita en
la semana 11.

> **Lo que mas se equivoca.** Los valores de `consumo_unitario` **ya vienen netos**: ya incluyen
> el rendimiento, la merma y la conversion de unidades. Si los vuelven a ajustar, el modelo les
> va a pedir mucho mas recurso del que existe y les va a salir **infactible**. Lean la columna
> `nota` de cada fila: ahi esta dicho que incluye cada coeficiente. En este caso el recurso que
> suele apretar es **las horas de instructor y la capacidad de salas**, pero eso lo tienen que demostrar ustedes con el
> modelo, no suponerlo.

In [ ]:
recursos = datos['recursos']

def tope(recurso, campo='limite'):
    """Tope del plan para un recurso: las filas con linea = 'todas'."""
    s = recursos[(recursos.recurso == recurso) & (recursos.linea == 'todas')][campo].dropna()
    return float(s.iloc[0]) if len(s) else None

def coef(recurso, campo='consumo_unitario'):
    """Coeficiente por linea: devuelve {linea: valor}, listo para usar en PuLP."""
    s = recursos[(recursos.recurso == recurso) & (recursos.linea != 'todas')]
    return s.set_index('linea')[campo].dropna().to_dict()

print('TOPES DEL PLAN  (linea = todas)')
display(recursos[recursos.linea == 'todas'][['recurso', 'coste', 'limite', 'unidad', 'nota']]
        .reset_index(drop=True))

print('\nCOEFICIENTES POR LINEA  (ya vienen netos: no los vuelva a ajustar)')
display(recursos[recursos.linea != 'todas']
        [['recurso', 'linea', 'disponibilidad', 'consumo_unitario', 'coste', 'margen', 'unidad']]
        .reset_index(drop=True))

Y la politica actual de la empresa, que es contra la que se compara la propuesta en la
semana 11. Reconstruyanla tal como esta escrita, **comprueben si cumple las restricciones** y,
si no las cumple, ajustenla de forma explicita y dejen registrado el ajuste. Que la politica
actual no sea factible es parte del ejercicio, no un error del enunciado.

In [ ]:
datos['politica_actual'][['regla_id', 'alcance', 'prioridad', 'cantidades', 'coste', 'descripcion']]

---
## Paso 5 · Guardar la base en un archivo

Hasta aqui la base vive en memoria y se pierde al cerrar el cuaderno. La guardamos en un
archivo y la reabrimos en **solo lectura**, que es la forma en que la va a consultar el asistente
de la semana 14: si el asistente solo puede leer, ninguna pregunta mal formulada puede borrar
una tabla. Es el mismo patron del LAB-D14.

In [ ]:
RUTA_BD = SALIDAS / f'{CASO}.duckdb'
if RUTA_BD.exists():
    RUTA_BD.unlink()

bd = duckdb.connect(str(RUTA_BD))
for t, df in datos.items():
    bd.register(f'_{t}', df)
    bd.execute(f'create table {t} as select * from _{t}')
bd.close()

# Se reabre en SOLO LECTURA. Asi se usara en la semana 14.
solo_lectura = duckdb.connect(str(RUTA_BD), read_only=True)
print('Base guardada en', RUTA_BD, '|', round(RUTA_BD.stat().st_size / 1e6, 1), 'MB')
solo_lectura.execute('''
    select 'clientes' as tabla, count(*) as filas from clientes
    union all select 'cohortes', count(*) from cohortes
    union all select 'demanda',  count(*) from demanda
''').df()

---
# Semana 9 · Clientes y valor economico

**Que decidir.** A quien contactar, con que umbral y cuanto vale economicamente hacerlo.

**Poblacion elegible.** Socios activos al corte con al menos 30 dias de antiguedad.

**Etiqueta de abandono.** Cancelacion o falta de renovacion durante los 30 dias siguientes al vencimiento que corresponda. Use `fecha_vencimiento` y **excluya los cortes sin seguimiento completo**.

**Que entregar.**

1. RFM y un CLV simplificado: contribucion mensual media historica x 6 meses. Declaren que esa
   aproximacion omite renovacion futura, descuento y variacion del margen.
2. Un modelo de churn con **scikit-learn**, con etiqueta y horizonte escritos explicitamente.
   Comparenlo contra una regla sencilla, para poder decir que el modelo aporta algo.
3. Matriz de confusion, precision, recall y PR-AUC. Una tabla o curva de umbrales que, para cada
   umbral, diga **cuantos contactos implica y cuanto dinero representa**.
4. El umbral elegido **con validacion**, no con prueba. La prueba se guarda para informar el
   desempeno final una sola vez.
5. Valor esperado incremental de contactar al cliente i:
   `probabilidad de churn_i x efecto incremental supuesto x CLV_i - coste del contacto - coste esperado del incentivo`.

**Tres cosas que cuestan puntos si se olvidan.**

- Cada predictor se calcula **solo con eventos anteriores a la fecha de corte**.
- Ningun cliente puede estar a la vez en entrenamiento, validacion y prueba.
- Incluyan el escenario de **efecto cero**: si la campana no evita ningun abandono, el valor
  incremental es negativo y lo correcto es no contactar. Un modelo de riesgo **no demuestra** que
  el contacto evite el abandono; eso es un supuesto del enunciado y hay que decirlo asi.

In [ ]:
# Trabajo de la semana 9
# La base ya esta en 'con'. Consulten con con.execute('select ...').df()


---
# Semana 10 · Pronostico de demanda

**Que pronosticar.** Reservas solicitadas por semana, sede y franja horaria, para cuatro semanas. Conserve la lista de espera: las asistencias observadas no son la demanda total, porque una franja llena deja de registrar a quien queria entrar.

**Que entregar.**

1. Un **modelo ingenuo obligatorio** (el valor de la semana anterior, o el de la misma semana del
   ano anterior) como piso de comparacion.
2. Al menos **un modelo estadistico** (statsmodels: suavizado exponencial, SARIMA o similar).
3. Backtesting temporal: reserven las **cuatro ultimas semanas** como prueba final y hagan al
   menos **tres ventanas de validacion** de cuatro semanas dentro del periodo anterior.
4. Error en MAE y WAPE. Si el denominador de WAPE es cero en alguna serie, lo reportan y usan MAE.
5. La traduccion a negocio: cuantas unidades hay que comprar, producir o programar.

**Lo que invalida el ejercicio.** Comparar metodos sobre semanas distintas, ajustar la imputacion
o el escalado con datos de prueba, y pronosticar sobre lo atendido en vez de lo solicitado.

In [ ]:
# Trabajo de la semana 10
# La base ya esta en 'con'. Consulten con con.execute('select ...').df()


---
# Semana 11 · Optimizacion con PuLP

**Que decidir.** a que socios ofrecer orientacion y cuantas sesiones programar por franja.

**Que entregar.**

1. Las **variables de decision** escritas con su significado y su unidad. Si son unidades
   indivisibles, son enteras (`cat='Integer'`).
2. La **funcion objetivo**: contribucion esperada neta, con los costes de intervencion dentro.
3. Las **restricciones**, una por recurso, cada una con su tope leido de `recursos.csv` con la
   funcion `tope()` de este cuaderno.
4. El **estado del solver**. Si dice `Infeasible`, no hay solucion y no se puede llamar optima a
   nada. Comprueben cada restriccion con tolerancia numerica despues de resolver.
5. La comparacion con la **politica actual**, bajo las mismas hipotesis, en margen y en uso de
   cada recurso.
6. Analisis de sensibilidad con los escenarios del enunciado.

**Los dos horizontes.** La retencion se valora a **seis meses** y la operacion a **cuatro
semanas**. Se reportan **por separado y no se suman**: los ingresos de un cliente contactado
estan en los dos calculos y sumarlos los contaria dos veces.

In [ ]:
# Trabajo de la semana 11
# La base ya esta en 'con'. Consulten con con.execute('select ...').df()


---
# Semana 12 · Procesos internos

**Que entregar.**

1. **Deteccion de anomalias** en la tabla `operaciones`, con la regla escrita y el umbral
   justificado.
2. Una **carta de control apropiada** al tipo de dato: definan unidad de observacion, periodo
   base, estadistico y supuestos. Con muestras de tamano constante va una carta de medias.
3. Un **responsable por cada alerta** y que se hace cuando salta.

**Lo que no se puede hacer.** Ajustar los limites con el mismo periodo que se quiere vigilar, ni
quitar puntos solo para que la base salga estable. Si no hay una base estable de mas de 30
periodos, lo documentan como limitacion y proponen revisarla antes de activar alertas.

**La distincion que se pregunta en la sustentacion.** Los limites de control **no son
especificaciones de tolerancia**. Una unidad fuera de especificacion no prueba que el proceso
este fuera de control, y un proceso fuera de control puede estar produciendo dentro de
especificacion. Y una anomalia detectada es una senal para revisar, **no evidencia de fraude**.

In [ ]:
# Trabajo de la semana 12
# La base ya esta en 'con'. Consulten con con.execute('select ...').df()


---
# Semana 13 · Gobierno y explicabilidad

**Que entregar.**

1. **Contrato de datos** y **linaje**: de donde sale cada tabla, quien la produce, con que
   frecuencia, quien accede y cuanto tiempo se conserva.
2. **Auditoria por subgrupos** del modelo de la semana 9: recall y falsos positivos por zona,
   canal y segmento, **siempre con el denominador a la vista**. Si un subgrupo tiene pocos casos,
   se informa la limitacion en vez de comparar tasas que no se sostienen.
3. **SHAP** sobre el modelo, con una lectura en palabras: que variable pesa y en que direccion.
4. **Matriz normativa** con fuentes oficiales vigentes: Ley 29733 y su reglamento, Ley 31814 y su
   reglamento (DS 115-2025-PCM). Sobre el **EU AI Act**, digan si aplica o no y por que.

**La regla que atraviesa el caso.** Un cliente sin `consentimiento_contacto` no puede ser
contactado, aunque el modelo lo ponga primero en la lista. Comprueben cuantos de los contactos
que propone la politica actual tienen consentimiento: puede que no sean todos.

In [ ]:
# Trabajo de la semana 13
# La base ya esta en 'con'. Consulten con con.execute('select ...').df()


---
# Semana 14 · Asistente analitico y checkpoint

**Que entregar.**

1. Un asistente **text-to-SQL de solo lectura** sobre la base que guardo el Paso 5 de este
   cuaderno, o un **RAG** sobre la tabla `documentos`.
2. Un **conjunto fijo de evaluacion** de 20 preguntas, definido *antes* de probar el asistente.
   No todas deben tener respuesta: incluyan preguntas ambiguas, preguntas sobre datos a los que el
   asistente no tiene acceso y alguna instruccion adversaria. Ante esas, **la respuesta correcta es
   abstenerse**.
3. **Tasa de acierto y tasa de alucinacion**, con la tabla pregunta por pregunta.
4. Una **linea base determinista** (consultas SQL escritas a mano) contra la que comparar. Si no
   llegaron a ejecutar un LLM, lo dicen: el LLM solo se reporta si de verdad se ejecuto.

**Esta semana se entrega el checkpoint (4 puntos del informe gerencial).**

In [ ]:
# Trabajo de la semana 14
# La base ya esta en 'con'. Consulten con con.execute('select ...').df()


---
# Semana 15 · Producto y comunicacion

**Que entregar.**

1. **Streamlit desplegado**, con enlace que funcione desde cualquier computadora: riesgo y
   umbral, demanda pronosticada, la optimizacion con sus escenarios, la carta de control, el
   gobierno y el asistente con su evaluacion a la vista.
2. **Informe gerencial**: la decision, la evidencia y lo que cuesta equivocarse. Para gerencia,
   no para el profesor.
3. **Hoja de ruta** analitica.
4. **Colab reproducible**: se ejecuta de principio a fin, sin rutas de la computadora de nadie,
   y da los mismos numeros. Por eso el Paso 0 fija la semilla y el Paso 2 usa un solo hilo.
5. **Ensayo cronometrado del pitch.** En la sustentacion hay 7 puntos grupales y 13 individuales:
   cada integrante tiene que poder defender cualquier parte del proyecto, no solo la suya.

In [ ]:
# Trabajo de la semana 15
# La base ya esta en 'con'. Consulten con con.execute('select ...').df()


---
## Antes de entregar

- [ ] Los diez controles del Paso 3 salen en verde, o la excepcion esta explicada.
- [ ] El cuaderno corre completo, de arriba abajo, en una sesion nueva de Colab.
- [ ] Ningun cliente aparece en dos tramos de la particion.
- [ ] El solver informa su estado y ninguna solucion infactible se llama optima.
- [ ] Los dos horizontes se reportan por separado.
- [ ] Esta el escenario de efecto cero.
- [ ] Ningun contacto propuesto carece de consentimiento.
- [ ] El enlace del Streamlit abre desde otra computadora.
- [ ] El fork del repositorio de datos es publico y tiene el trabajo dentro.